# Gene intersection

This is the notebook counterpart of `scripts/01_gene_intersection.py`.
It maps cell line gene symbols to Ensembl IDs, reads the tumor feature
table, and reports the shared gene set. The script remains the
authoritative writer of `data/processed/genes_combined.csv`.

The notebook imports the same functions as the script, so the mapping
logic has one implementation.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import gene_intersection as gi
from src.utils import load_config

config = load_config()
raw = config["paths"]["raw_data"]
processed = config["paths"]["processed"]

gene_anno_path = ROOT / raw["gene_anno"]
cell_line_path = ROOT / raw["cell_line"]
tumor_path = ROOT / raw["tumor"]
output_path = ROOT / processed["genes_combined"]

In [2]:
gene_anno = gi.load_gene_anno(gene_anno_path)
cell_line_genes = gi.load_and_prepare_cell_line_genes(cell_line_path, gene_anno)
tumor_genes = gi.load_and_prepare_tumor_genes(tumor_path)
genes = gi.combine_genes(cell_line_genes, tumor_genes, gene_anno)

print(f"cell line genes mapped: {len(cell_line_genes):,}")
print(f"tumor genes: {len(tumor_genes):,}")
print(f"shared genes: {len(genes):,}")
print(f"unique gene ids: {genes['gene_id'].is_unique}")
genes.head()

cell line genes mapped: 28,697
tumor genes: 43,050
shared genes: 28,617
unique gene ids: True


,gene_id,gene_symbol
0,ENSG00000121410,A1BG
1,ENSG00000268895,A1BG-AS1
2,ENSG00000148584,A1CF
3,ENSG00000175899,A2M
4,ENSG00000166535,A2ML1


The script writes the table to `data/processed/genes_combined.csv`. Run
`./run_preprocessing.sh` for the full pipeline; this notebook is for
inspection and small checks.